In [1]:
import numpy as np



def FT_odd(N,cutoff):
    """
    Compute Fourier transform coefficients for odd N

    Parameters:
        N: integer, must be odd
    Returns:
        coeffs: NumPy array containing computed results
    """
    hbar_c = 197.3269804                          
    a_fm = 197.3269804 / cutoff             
    m = 938.92              

    c = 2 * (np.pi ** 2) * (hbar_c ** 2) / ((N ** 3) * m * (a_fm ** 2))

    coeffs = np.zeros(int(N / 2) + 1)

    for delta_j in range(int(N / 2) + 1):
        S = 0
        for n in range(-int((N-1)/2), int((N+1)/2)):
            S += n ** 2 * np.exp(1j * 2 * np.pi * delta_j * n / N)

        coeffs[delta_j] = c * S.real

    return coeffs




In [2]:
from quspin.basis import boson_basis_general                         
import numpy as np         
from quspin.operators import hamiltonian           

L = 3                                
N_3d = L * L * L            


coeffs = FT_odd(N=3,cutoff=170)

c_2 = -143.2028868560966
c_3 = 107.232753

s = np.arange(N_3d)               
x = s % L                            
y = (s // L) % L                        
z = s // (L * L)                         

S_x = x + L * (L - 1 - z) + L * L * y

T_x = (x + 1) % L + L * y + L * L * z           
T_y = x + L * ((y + 1) % L) + L * L * z           
T_z = x + L * y + L * L * ((z + 1) % L)           

P_x = (L - x - 1) + L * y + L * L * z              
P_y = x + L * (L - y - 1) + L * L * z              
P_z = x + L * y + L * L * (L - z - 1)              

def Tx(s, i):
    """Coordinate transform for translation by i sites along x (for odd L)"""
    x = s % L
    y = (s // L) % L
    z = s // (L * L)
    return int((x + i) % L + L * y + L * L * z)

def Ty(s, i):
    """Coordinate transform for translation by i sites along y (for odd L)"""
    x = s % L
    y = (s // L) % L
    z = s // (L * L)
    return int(x + L * ((y + i) % L) + L * L * z)

def Tz(s, i):
    """Coordinate transform for translation by i sites along z (for odd L)"""
    x = s % L
    y = (s // L) % L
    z = s // (L * L)
    return int(x + L * y + L * L * ((z + i) % L))

R_1 = y + z * L + x * L * L                          

basis_2 = boson_basis_general(
    N_3d,                        
    Nb=2,                        
    kxblock=(T_x, 0),                          
    kyblock=(T_y, 0),                
    kzblock=(T_z, 0),                
    pxblock=(P_x, 0),                
    pyblock=(P_y, 0),                
    pzblock=(P_z, 0),                
    sxblock=(S_x, 0),                 
    r1block=(R_1, 0)                  
)

Ns = basis_2.Ns          
print(f"Basis-space dimension: {Ns}")

basis_3 = boson_basis_general(
    N_3d,                        
    Nb=3,                        
    kxblock=(T_x, 0),                          
    kyblock=(T_y, 0),                
    kzblock=(T_z, 0),                
    pxblock=(P_x, 0),                
    pyblock=(P_y, 0),                
    pzblock=(P_z, 0),                
    sxblock=(S_x, 0),                 
    r1block=(R_1, 0)                  
)

Ns = basis_3.Ns          
print(f"Basis-space dimension: {Ns}")

basis_4 = boson_basis_general(
    N_3d,                        
    Nb=4,                        
    kxblock=(T_x, 0),                          
    kyblock=(T_y, 0),                
    kzblock=(T_z, 0),                
    pxblock=(P_x, 0),                
    pyblock=(P_y, 0),                
    pzblock=(P_z, 0),                
    sxblock=(S_x, 0),                 
    r1block=(R_1, 0)                  
)

Ns = basis_4.Ns          
print(f"Basis-space dimension: {Ns}")


pot = [[coeffs[0] * 3, i] for i in range(N_3d)]

interact = [[0.5 * c_2, i, i, i, i] for i in range(N_3d)]

three_body = [[c_3 / 6, i, i, i, i, i, i] for i in range(N_3d)]

static = [['n', pot], ["++--", interact], ['+++---', three_body]]

max_k = int((L + 1) / 2)                
for k in range(1, max_k):
    hopping_terms = (
        [[coeffs[k], i, Tx(i, k)] for i in range(N_3d)] +           
        [[coeffs[k], i, Ty(i, k)] for i in range(N_3d)] +           
        [[coeffs[k], i, Tz(i, k)] for i in range(N_3d)]             
    )
    static.append(['+-', hopping_terms])
    static.append(['-+', hopping_terms])






Basis-space dimension: 4
Basis-space dimension: 14
Basis-space dimension: 58


/home/lzs/lzsenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3699: GeneralBasisWarning: using non-commuting symmetries can lead to unwanted behaviour of general basis, make sure that quantum numbers are invariant under non-commuting symmetries!
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
H_2 = hamiltonian(
    static, [],
    basis=basis_2,
    dtype=np.float64,
    check_herm=False,           
    check_symm=False,           
    check_pcon=False              
)

H_sparse = H_2.tocsr()
print(f"Number of nonzero Hamiltonian elements: {len(H_sparse.data)}")

E_GS, V_GS = H_2.eigsh(k=1, which='SA')                                   

print(f"Ground-state energy: {E_GS[0]}")


Number of nonzero Hamiltonian elements: 10
Ground-state energy: -11.650256388464072


In [4]:
H_3 = hamiltonian(
    static, [],
    basis=basis_3,
    dtype=np.float64,
    check_herm=False,           
    check_symm=False,           
    check_pcon=False              

)

H_sparse = H_3.tocsr()
print(f"Number of nonzero Hamiltonian elements: {len(H_sparse.data)}")

E_GS, V_GS = H_3.eigsh(k=1, which='SA')                                   

print(f"Ground-state energy: {E_GS[0]}")


Number of nonzero Hamiltonian elements: 66
Ground-state energy: -35.835796099900975


In [5]:
H_4 = hamiltonian(
    static, [],
    basis=basis_4,
    dtype=np.float64,
    check_herm=False,           
    check_symm=False,           
    check_pcon=False              
)

H_sparse = H_4.tocsr()
print(f"Number of nonzero Hamiltonian elements: {len(H_sparse.data)}")

E_GS, V_GS = H_4.eigsh(k=1, which='SA')                                   

print(f"Ground-state energy: {E_GS[0]}")


Number of nonzero Hamiltonian elements: 532
Ground-state energy: -68.8157756211217
